In [ ]:
from config import EXTERNAL_DATASET_INPUTS, INTERNAL_DATASET_INPUTS, Columns, ALIGNMENT_PATH
from data_processing.processing import TextProcessor
from utils import stack_csvs_from_folder, drop_empty_rows
import pandas as pd
import re
from alignment_interactive import process_dataframe, BatchReviewer, build_final_dataset
from transformers import AutoTokenizer
import html
from IPython.display import display, HTML

In [27]:
df = pd.read_csv("../scrapers/url_analysis/splitted_final_scraped_ai.csv")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16") 
           
df = df.dropna()

def tokenize(text):
    return tokenizer.encode(text, add_special_tokens=True)

df['transliteration_t'] = df["transliteration"].apply(tokenize)
df['translation_t'] = df["translation"].apply(tokenize)



In [29]:
print(f"Max tokens found: {df['transliteration_t'].str.len().max()}")
print(f"Max tokens found: {df['translation_t'].str.len().max()}")

print(f"Mean tokens found transliteration: {df['transliteration_t'].str.len().mean()}")
print(f"Mean tokens found translation: {df['translation_t'].str.len().mean()}")

Max tokens found: 400
Max tokens found: 623
Mean tokens found transliteration: 78.31774872590294
Mean tokens found translation: 38.64300068002781


In [7]:
condition = (df["translit_tokens"].str.len() > 512) | (df["transl_tokens"].str.len() > 512)

# Filter the dataframe
long_entries = df[condition]

print(f"Found {len(long_entries)} rows where either translation or translation > 512 tokens.")

KeyError: 'translit_tokens'

In [53]:
condition = (df["translation"].str.len() > 500) | (df["transliteration"].str.len() > 500)

# Filter the dataframe
long_entries = df[condition]

print(f"Found {len(long_entries)} rows where either translation or translation > 500 characters.")

Found 571 rows where either translation or translation > 500 characters.


In [54]:
cond1 = (df["transliteration"].str.len() > 500) & (df["transliteration"].str.len() < 1000)
cond2 = (df["eng"].str.len() > 500) & (df["eng"].str.len() < 1000)

condition = cond1 | cond2
# Filter the dataframe
long_entries = df[condition]

print(f"Found {len(long_entries)} rows where either translation or translation where number characters is in (1000-500)")

Found 570 rows where either translation or translation where number characters is in (1000-500)


In [30]:

def compute_length_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Add word-count columns for transliteration and translation."""
    df = df.copy()
    df[Columns.TRANSLITERATION + "len"] = df[Columns.TRANSLITERATION + "_t"].apply(
        lambda x: len(x)
    )
    df[Columns.TRANSLATION + "len"] = df[Columns.TRANSLATION+ "_t"].apply(lambda x: len(x))

    df["len_ratio"] = (
        df[Columns.TRANSLATION + "len"] / df[Columns.TRANSLITERATION + "len"]
    )
    return df

def print_length_summary(df: pd.DataFrame):
    print("\n--- Length Summary ---")
    print(f"  Total rows:              {len(df)}")
    print(
        f"  Avg transliteration len: {df[Columns.TRANSLITERATION + 'len'].mean():.1f} words"
    )
    print(
        f"  Avg translation len:     {df[Columns.TRANSLATION + 'len'].mean():.1f} words"
    )
    print(f"  Avg len ratio (t/t):     {df['len_ratio'].mean():.2f}")
    print(f"  Std len ratio:           {df['len_ratio'].std():.2f}")


def find_suspicious_rows(df: pd.DataFrame, z_threshold: float = 2.0) -> pd.DataFrame:
    """
    Flag rows where the length ratio is more than z_threshold standard deviations
    from the mean. These are likely misaligned or garbage pairs.
    """
    mean = df["len_ratio"].mean()
    std = df["len_ratio"].std()

    lower = mean - z_threshold * std
    upper = mean + z_threshold * std

    suspicious = df[(df["len_ratio"] < lower) | (df["len_ratio"] > upper)].copy()
    suspicious["z_score"] = (suspicious["len_ratio"] - mean) / std

    print(f"\n--- Suspicious Rows (z > {z_threshold}) ---")
    print(f"  Mean ratio:   {mean:.2f}")
    print(f"  Lower bound:  {lower:.2f}")
    print(f"  Upper bound:  {upper:.2f}")
    print(
        f"  Flagged rows: {len(suspicious)} / {len(df)}  ({len(suspicious) / len(df):.1%})"
    )

    return suspicious.sort_values("z_score", key=abs, ascending=False)


def print_examples(suspicious: pd.DataFrame, n: int = 10):
    """Print the most extreme misaligned pairs for manual inspection."""
    print(f"\n--- Top {n} Most Misaligned Pairs ---")
    for _, row in suspicious.head(n).iterrows():
        print(f"  Ratio: {row['len_ratio']:.2f}  (z={row['z_score']:.2f})")
        print(
            f"  Translit [{row[Columns.TRANSLITERATION + 'len']} words]: {str(row[Columns.TRANSLITERATION])[:120]}"
        )
        print(
            f"  Transl   [{row[Columns.TRANSLATION + 'len']} words]: {str(row[Columns.TRANSLATION])[:120]}"
        )
        print()


def length_distribution(df: pd.DataFrame, buckets=(0.2, 0.5, 1.0, 2.0, 3.0, 5.0)):
    """Show what fraction of pairs fall in each ratio bucket."""
    print("\n--- Ratio Distribution ---")
    prev = 0.0
    for b in buckets:
        count = ((df["len_ratio"] > prev) & (df["len_ratio"] <= b)).sum()
        print(f"  {prev:.1f} < ratio <= {b:.1f} : {count:>6}  ({count / len(df):.1%})")
        prev = b
    count = (df["len_ratio"] > prev).sum()
    print(f"  ratio > {prev:.1f}           : {count:>6}  ({count / len(df):.1%})")





In [ ]:
df = compute_length_stats(df)

# Summary stats
print_length_summary(df)

# Distribution across ratio buckets
length_distribution(df)

# Find suspicious rows
suspicious = find_suspicious_rows(df, z_threshold=2.0)

# Print the worst for manual inspection
print_examples(suspicious, n=10)



--- Length Summary ---
  Total rows:              130877
  Avg transliteration len: 78.3 words
  Avg translation len:     38.6 words
  Avg len ratio (t/t):     0.53
  Std len ratio:           0.29

--- Ratio Distribution ---
  0.0 < ratio <= 0.2 :   1079  (0.8%)
  0.2 < ratio <= 0.5 :  63431  (48.5%)
  0.5 < ratio <= 1.0 :  65087  (49.7%)
  1.0 < ratio <= 2.0 :   1140  (0.9%)
  2.0 < ratio <= 3.0 :     44  (0.0%)
  3.0 < ratio <= 5.0 :     50  (0.0%)
  ratio > 5.0           :     46  (0.0%)

--- Suspicious Rows (z > 2.0) ---
  Mean ratio:   0.53
  Lower bound:  -0.05
  Upper bound:  1.12
  Flagged rows: 894 / 130877  (0.7%)

--- Top 10 Most Misaligned Pairs ---
  Ratio: 26.31  (z=88.02)
  Translit [13 words]: i-szi2-im su2-a-tim?
  Transl   [342 words]: 'the suckling of the suckling of the suckling of the suckling of the suckling of the suckling of the suckling of the suc

  Ratio: 26.31  (z=88.02)
  Translit [13 words]: i-szi2-im su2-a-tim?
  Transl   [342 words]: 'the suckling of th

In [10]:
df.head()

,Unnamed: 0,id,translation,transliteration,transliteration_t,translation_t,transliterationlen,translationlen,len_ratio
0,4,5train_akkademia,"I dug out the Patti-Enlil canal, which had lai...",{ID₂}-pa-at-ti-{d}-EN.LIL₂ ša ul-tu UD.MEŠ ru-...,"[250004, 10666, 7146, 250321, 249, 9, 257, 9, ...","[250004, 87, 58709, 1810, 70, 9876, 118, 9, 77...",41,24,0.585366
1,5,6train_akkademia,", and I made an abundance of water gurgle thro...",ah-re-e-ma i-na qer-be₂-e-ša₂ u₂-šah-bi-ba A.M...,"[250004, 1263, 9, 107, 9, 13, 9, 192, 17, 9, 7...","[250004, 6, 4, 136, 87, 7228, 142, 130807, 395...",38,17,0.447368
2,8,9train_akkademia,"thousand to the province of the turtānu, 10,00...",LIM NAM {LU₂}-tur-ta-ni 10 LIM NAM {LU₂}-NIMGI...,"[250004, 6, 83318, 75033, 10666, 22335, 250321...","[250004, 199823, 47, 70, 140280, 111, 70, 2130...",27,25,0.925926
3,9,10train_akkademia,thousand (to) the province of the chief cupbea...,LIM NAM {LU₂}-GAL.BI.LUL {LU₂}-GAL.BI.LUL,"[250004, 6, 83318, 75033, 10666, 22335, 250321...","[250004, 199823, 15, 188, 16, 70, 140280, 111,...",23,16,0.695652
4,13,14train_akkademia,I received as his payment,10 GUN KU₃.GI i-na KAL-te 1 LIM GUN KU₃.BABBAR,"[250004, 209, 6, 96102, 8992, 250322, 42887, 1...","[250004, 87, 75204, 237, 1919, 81997, 2]",23,7,0.304348


In [ ]:
# Calculate the bounds
upper_bound = mean_ratio + (2 * std_ratio)
lower_bound = max(0, mean_ratio - (2 * std_ratio)) # Ensure we don't go below 0

# Create a new DataFrame keeping only the rows within the bounds
df2 = df1[(df1['transliterationlen'] >= lower_bound) & (df1['transliterationlen'] <= upper_bound)]

print(f"Original rows: {len(df1)}")
print(f"Filtered rows: {len(df2)}")
print(f"Dropped rows: {len(df1) - len(df2)}")


mean_ratio = df2['translationlen'].mean() 
std_ratio = df2['translationlen'].std()   
# Calculate the bounds
upper_bound = mean_ratio + (2 * std_ratio)
lower_bound = max(0, mean_ratio - (2 * std_ratio)) # Ensure we don't go below 0

# Create a new DataFrame keeping only the rows within the bounds
df3 = df2[(df2['translationlen'] >= lower_bound) & (df2['translationlen'] <= upper_bound)]

print(f"Original rows: {len(df2)}")
print(f"Filtered rows: {len(df3)}")
print(f"Dropped rows: {len(df2) - len(df3)}")



Original rows: 73354
Filtered rows: 71125
Dropped rows: 2229
Original rows: 71125
Filtered rows: 67501
Dropped rows: 3624
Original rows: 67501
Filtered rows: 63931
Dropped rows: 3570


In [24]:
print_length_summary(df3)
print(f"Max tokens found: {df3['transliteration_t'].str.len().max()}")
print(f"Max tokens found: {df3['translation_t'].str.len().max()}")

suspicious = find_suspicious_rows(df3, z_threshold=2.0)

df3.to_csv("cleaned_splitted_scraped_human")
# Print the worst offenders for manual inspection
print_examples(suspicious, n=10)



--- Length Summary ---
  Total rows:              63931
  Avg transliteration len: 36.2 words
  Avg translation len:     16.3 words
  Avg len ratio (t/t):     0.49
  Std len ratio:           0.21
Max tokens found: 147
Max tokens found: 48

--- Suspicious Rows (z > 2.0) ---
  Mean ratio:   0.49
  Lower bound:  0.06
  Upper bound:  0.91
  Flagged rows: 3020 / 63931  (4.7%)

--- Top 10 Most Misaligned Pairs ---
  Ratio: 1.17  (z=3.21)
  Translit [29 words]: [...] szA-ni-tum# [x x x] ki-i tu-szar-szi-du-usz
  Transl   [34 words]: … "A second time" = Marduk II 64 means …. "You lay his foundations" cp. Marduk II l. 65,

  Ratio: 1.17  (z=3.21)
  Translit [29 words]: _be gir 150 z_É _szub_—_asz-te gar_-_mesz masz silim_-im
  Transl   [34 words]: The 'path' on the left of the gall bladder and the 'base of the throne' are present. The 'increment' is normal.

  Ratio: 1.17  (z=3.21)
  Translit [29 words]: [x x x] mar-ka-si-szu2-nu mu-nam#-[me-rat ...]
  Transl   [34 words]: . . . their i.e., th

In [32]:
mean = df1['transliterationlen'].mean()
std = df1['transliterationlen'].std()

lower_bound = mean - 2 * std
upper_bound = mean + 2 * std

df2 = df1[(df1['transliterationlen'] >= lower_bound) & (df1['transliterationlen'] <= upper_bound)]

print(f"Removed {len(df1) - len(df2)} rows ({(len(df1) - len(df2)) / len(df1) * 100:.1f}%)")
print(f"Bounds: {lower_bound:.1f} — {upper_bound:.1f} characters")

Removed 3624 rows (5.1%)
Bounds: -49.6 — 147.6 characters


In [33]:
df2.to_csv("cleaned_splitted_scraped_ai.csv")


In [ ]:
df3.to_csv("akkademia_cleaned2.csv")

In [ ]:
def extract_subset_symmetrical(a, b):
    words_a = a.split()
    if not words_a:
        return None
    
    start_anchor = ""
    for i in range(1, len(words_a) + 1):
        candidate_start = " ".join(words_a[:i])
        # Check uniqueness in A
        if len(re.findall(rf'\b{re.escape(candidate_start)}\b', a)) == 1:
            start_anchor = candidate_start
            break
            
    end_anchor = ""
    for j in range(1, len(words_a) + 1):
        candidate_end = " ".join(words_a[-j:])
        # Check uniqueness in A
        if len(re.findall(rf'\b{re.escape(candidate_end)}\b', a)) == 1:
            end_anchor = candidate_end
            break

    # Fallback: if uniqueness isn't found, use first/last words
    start_anchor = start_anchor or words_a[0]
    end_anchor = end_anchor or words_a[-1]

    # use (.*?) for a non-greedy match between the two unique anchors
    pattern = rf'\b{re.escape(start_anchor)}\b.*?\b{re.escape(end_anchor)}\b'
    match = re.search(pattern, b, re.IGNORECASE | re.DOTALL)
    
    return match.group(0) if match else None

text_a = "the small brown dog"
text_b = "I saw the very small ugly ass brown dog in the park"

print(f"Match found: {extract_subset_symmetrical(text_a, text_b)}")

text_a_alt = "the dog saw the red dog"
text_b_alt = "In the woods, the dog saw the blue red dog and barked"
# It should anchor to 'red dog' instead of just 'dog'
print(f"Match found: {extract_subset_symmetrical(text_a_alt, text_b_alt)}")

Match found: the very small ugly ass brown dog
Match found: the dog saw the blue red dog


In [10]:
    def slice_transliteration(source_text, alignment_rows):
        """
        Use the word indices from alignment data to cut the transliteration.
        """
        words = source_text.split()
        sliced_sources = []

        for i in range(len(alignment_rows)):
            curr_start_idx = alignment_rows.iloc[i]
            next_idx = (
                alignment_rows.iloc[i + 1] if i + 1 < len(alignment_rows) else None
            )  # none at last part

            if pd.isna(curr_start_idx["first_word_number"]):
                # check: csv doesn't have a word number for this sentence, can't split it
                sliced_sources.append("")
                continue

            start_index = int(curr_start_idx["first_word_number"]) - 1

            if next_idx is not None and not pd.isna(next_idx["first_word_number"]):
                end_index = int(next_idx["first_word_number"]) - 1
            else:
                end_index = len(words)

            sliced_sources.append(" ".join(words[start_index:end_index]))

        return sliced_sources


In [ ]:
def align(row, guide_by_id,i):
    oare_id = row["id"]
    if oare_id not in guide_by_id:
        doc_row = row.to_dict(); doc_row["level"] = "document"; doc_row["alignment_status"] = "NO_GUIDE"
        return [doc_row]
    

    sent_rows = guide_by_id[oare_id]
    
    sliced_trans = slice_transliteration(row["transliteration"], sent_rows)
    
    final_segments = []
    success_count = 0

    words_trans = row["transliteration"].split()

    for i, guide_row in sent_rows.iterrows():
        extracted_translation = extract_subset_symmetrical(
            guide_row["translation"], 
            row["translation"]
        )
        
        if extracted_translation is None:
            leftover_trans = " ".join(words_trans[success_count:])
            
            # keep the original full translation to be safe 
            leftover_row = row.to_dict().copy()
            leftover_row["transliteration"] = leftover_trans
            leftover_row["translation"] = row["translation"] 
            leftover_row["level"] = "document"
            leftover_row["alignment_status"] = f"PARTIAL_BREAK_AT_SENT_{i}"
            
            final_segments.append(leftover_row)
            return final_segments 
        
        segment = row.to_dict().copy()
        segment["transliteration"] = sliced_trans[i]
        segment["translation"] = extracted_translation
        segment["level"] = "sentence"
        segment["sentence_idx"] = i
        segment["alignment_status"] = "SUCCESS"
        
        final_segments.append(segment)
        success_count += 1 

    return final_segments

In [24]:

i = 0
sentences_oare = pd.read_csv(ALIGNMENT_PATH)
sentences_oare = sentences_oare.dropna(subset=['translation'])
train_df = load_data()

guide_by_id = {
    k: g.sort_values("first_word_number").reset_index(drop=True)
    for k, g in sentences_oare.groupby("text_uuid")
}

all_rows = []

for _, row in train_df.iterrows():
    processed_rows = align(row, guide_by_id,i)
    all_rows.extend(processed_rows)

final_df = pd.DataFrame(all_rows)

# Quick check on the results
print(final_df["level"].value_counts())
print(i)

No empty rows found.
Rows before: 1561, Rows after: 1561
level
document    1558
sentence      14
Name: count, dtype: int64
0


In [13]:
condition = (final_df[Columns.TRANSLITERATION].str.len() > 500) | (final_df[Columns.TRANSLATION].str.len() > 500)

# Filter the dataframe
long_entries = df[condition]

print(f"Found {len(long_entries)} rows where either translation or translation > 500 characters.")

Found 625 rows where either translation or translation > 500 characters.


/tmp/ipykernel_117250/418952698.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  long_entries = df[condition]


In [17]:
sentences_df = final_df[final_df["level"] == "sentence"]
print(len(sentences_df))


14


In [15]:
df = compute_length_stats(final_df)

# Summary stats
print_length_summary(df)

# Distribution across ratio buckets
length_distribution(df)

# Find suspicious rows
suspicious = find_suspicious_rows(df, z_threshold=2.0)

# Print the worst offenders for manual inspection
print_examples(suspicious, n=10)



--- Length Summary ---
  Total rows:              1572
  Avg transliteration len: 426.7 words
  Avg translation len:     496.2 words
  Avg len ratio (t/t):     1.08
  Std len ratio:           0.50

--- Ratio Distribution ---
  0.0 < ratio <= 0.2 :     40  (2.5%)
  0.2 < ratio <= 0.5 :     79  (5.0%)
  0.5 < ratio <= 1.0 :    345  (21.9%)
  1.0 < ratio <= 2.0 :   1075  (68.4%)
  2.0 < ratio <= 3.0 :     20  (1.3%)
  3.0 < ratio <= 5.0 :     11  (0.7%)
  ratio > 5.0           :      2  (0.1%)

--- Suspicious Rows (z > 2.0) ---
  Mean ratio:   1.08
  Lower bound:  0.09
  Upper bound:  2.08
  Flagged rows: 40 / 1572  (2.5%)

--- Top 10 Most Misaligned Pairs ---
  Ratio: 9.25  (z=16.47)
  Translit [289 words]: 4 GÚ AN.NA ku-nu-ki ša a-šùr-i-mì-tí {d}IM-ba-ni ir-dí-a-ni nu-sà-ni-iq-šu-ma 4 GÚ 6 ma-na 5 GÍN AN.NA ni-is-ni-iq ŠÀ.BA
  Transl   [2672 words]: 6 minas of refined silver is owed by Kulaba, Šupun-ahšu and hanunu. 6 minas of refined silver is owed by Bēlum-bāni son 

  Ratio: 8.88  (

In [662]:
df = pd.read_csv("../dataset/default_train.csv", on_bad_lines='warn')

        

In [663]:
purple_style = "background-color: #e8b4fe; padding: 2px 4px; border-radius: 3px;"


In [ ]:
def escape_gaps(text):
    gaps = re.findall(r'<[^>]+>', text)
    placeholder = "GAPTOKEN{}"
    for i, gap in enumerate(gaps):
        text = text.replace(gap, placeholder.format(i), 1)

    text = html.escape(text)

    for i, gap in enumerate(gaps):
        text = text.replace(placeholder.format(i), gap)
        text = html.escape(text,quote=False)

    return text

In [36]:
def merge_close_spans(html, match_style, merge_style, gap_size=6):
    gap = rf'((?:\s*\S+\s*){{1,{gap_size}}})'
    blue = re.escape(match_style)
    green = re.escape(merge_style)

    def safe_merge(m):
        gap_text = m.group(3)
        first_span_content = m.group(2)
        if re.search(r'\d|\.', gap_text) or first_span_content.rstrip().endswith('.'):
            return m.group(0)
        return f'<span style="{merge_style}">{m.group(2)}{gap_text}{m.group(5)}</span>'
    while True:
        p1 = rf'(<span style="{blue}">(.*?)</span>){gap}(<span style="{blue}">(.*?)</span>)'
        p2 = rf'(<span style="{green}">(.*?)</span>){gap}(<span style="{blue}">(.*?)</span>)'
        p3 = rf'(<span style="{green}">(.*?)</span>){gap}(<span style="{green}">(.*?)</span>)'
        new_html = html
        for p in [p1, p2, p3]:
            new_html = re.sub(p, safe_merge, new_html, flags=re.DOTALL)
        if new_html == html:
            break
        html = new_html
    return html

In [ ]:
blue_style = 'background-color: rgba(173, 216, 230, 0.4); border: 1px solid #ADD8E6; padding: 2px; border-radius: 3px;'
red_style  = 'background-color: rgba(255, 100, 100, 0.4); border: 1px solid #FF6464; padding: 2px; border-radius: 3px;'
green_style = 'background-color: rgba(144, 238, 144, 0.4); border: 1px solid #90EE90; padding: 2px; border-radius: 3px;'


def highlight_seal_of(text):
    name_chars = r"[\w\-\.\{\}\(\)\u00c0-\u00ff\u0161\u0160\u1e2b\u1e63\u1e6d\u0101\u0100\u012b\u012a\u016b\u016a\u0113ù']+"
    name_chars = r"[^\s]+"
    # AKK
    conj_akk = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")?"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"

    # ENGLISH
    relation_eng = r"(?:[,\s]+(?:s\.|son|brother|sister|daughter|wife)\s+of\s+" + name_chars + r")?"
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!seal\s+of)" + name_chars + r")?"
    eng_unit     = rf"seal\s+of\s+{name_chars}{relation_eng}{conj_eng}"
    eng_pattern  = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?seal\s+of(?:\s+(?![,;]){name_chars})?{relation_eng}{conj_eng})*[,;]?"


    try:
        
        highlighted_text = re.sub(
            rf"({akk_pattern})",
            lambda m: f'<span style="{blue_style}">{m.group(1)}</span>',
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        
        eng_full = rf"(^|[.,;:]\s*)({eng_pattern})"        
        highlighted_text = re.sub(
            eng_full,
            lambda m: m.group(1) + f'<span style="{blue_style}">{m.group(2)}</span>',
            highlighted_text,
            flags=re.IGNORECASE | re.MULTILINE
        )
        
        parts = re.split(r'(<span[^>]*>.*?</span>)', highlighted_text, flags=re.DOTALL)
        result_parts = []
        for part in parts:
            if part.startswith('<span'):
                result_parts.append(part)
            else:
                result_parts.append(re.sub(
                    r'(seal\s+of)',
                    rf'<span style="{red_style}">\1</span>',
                    part,
                    flags=re.IGNORECASE
                ))
        highlighted_text = ''.join(result_parts)

        highlighted_text = merge_close_spans(highlighted_text, blue_style, green_style)
        display(HTML(f"<div style='line-height: 1.8; font-family: sans-serif; white-space: pre-wrap;'>{highlighted_text}</div>"))

    except re.error as e:
        print(f"Regex Error: {e}")

In [ ]:

def highlight_sealed_by(text):
    text = html.escape(text)

    name_chars = r"[^\s,]+"
    
    # AKK
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")*"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    # ENGLISH
    relation_eng = r"(?:[,\s]+(?:(?:his|her)\s+(?:brother|sister)|the\s+scribe|(?:s\.|son|brother|sister|daughter|wife|\(grand\)son|grandson)\s+of\s+" + name_chars + r"))?"    
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!sealed\s+by)" + name_chars + r")?"
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_chars + r")?"

    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister)\s+)?"
    eng_unit = rf"sealed\s+by\s+{possessive_prefix}{name_chars}{relation_eng}{conj_eng}"
    eng_pattern = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?(?:sealed\s+)?by\s+{possessive_prefix}(?:(?![,;]){name_chars})?{relation_eng}{conj_eng})*{trailing_collective}[,;]?"

    eng_full = rf"(^\s*|[,;:]\s*)({eng_pattern})"
    try:
        highlighted_text = re.sub(
            rf"({akk_pattern})",
            lambda m: f'<span style="{blue_style}">{m.group(1)}</span>',
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        highlighted_text = re.sub(
            eng_full,
            lambda m: m.group(1) + f'<span style="{blue_style}">{m.group(2)}</span>',
            highlighted_text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        parts = re.split(r'(<span[^>]*>.*?</span>)', highlighted_text, flags=re.DOTALL)
        result_parts = []
        for part in parts:
            if part.startswith('<span'):
                result_parts.append(part)
            else:
                result_parts.append(re.sub(
                    r'(sealed\s+by)',
                    rf'<span style="{red_style}">\1</span>',
                    part,
                    flags=re.IGNORECASE
                ))
        highlighted_text = ''.join(result_parts)

        highlighted_text = merge_close_spans(highlighted_text, blue_style, green_style)
        display(HTML(f"<div style='line-height: 1.8; font-family: sans-serif; white-space: pre-wrap;'>{highlighted_text}</div>"))

    except re.error as e:
        print(f"Regex Error: {e}")

In [ ]:
def highlight_witnessed_by(text):
    import unicodedata
    text = unicodedata.normalize('NFC', text)
    name_chars = r"[^\s,]+"
    # AKK
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"

    gap_token = r"(?:\s*<[^>]*>(?:\s+(?!IGI\b|DUMU\b|KIŠIB\b|KÙ\.BABBAR\b|witnessed\b|by\b)[^\s<]+)?)*"
    relation_akk = (
        r"(?:\s+(?:DUMU|a-ḫu-ú)\s+[^\s,;<]+(?:\s*<[^>]*>)*"
        r"(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|a-ḫa-ma\b|a-na\b|i-na\b|§)[^\s,;]*-[^\s,;]+"
        r"(?:\s*<[^>]*>)*)*)*"
    )
    akk_name = r"[^\s]*[-\.][^\s]*[-\.][^\s]+|[^\s]*-[^\s]+(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|§)[^\s]*-[^\s]+)*|[^\s]*\.[^\s]+"

    akk_igi_unit = rf"IGI\s+(?!GÍR\b|a-we-li\b)(?:(?:{akk_name}){gap_token}(?:{relation_akk})|(?:DUMU|a-ḫu-ú)\s+{name_chars}{gap_token}{relation_akk}){conj_akk}\s*(?!\d)"
    
    akk_igi_multi  = rf"(?:{akk_igi_unit}\s*){{2,}}"
    akk_igi_single = rf"{akk_igi_unit}$"
    akk_igi        = rf"(?:{akk_igi_multi}|{akk_igi_single})"
    
    akk_kisib      = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    akk_pattern    = rf"(?:{akk_kisib}|{akk_igi})"

    
    # ENGLISH

    title_eng    = r"(?:,\s*the\s+[^\s,;.]+)?"

    occupations = r"(?:packer|scribe|merchant|smith|weaver|potter|herald|priestess)"


    #gap_token = r"(?:\s*(?:<|&lt;)[^<>&;]*(?:>|&gt;))*"
    name_or_gap = rf"(?:{name_chars}|{gap_token})"
    possessive_witness = r"(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife|partner)"    
        
    relation_eng = (
        r"(?:[,\s]+(?:"
        r"(?:his|her)\s+(?:brother|sister|son|daughter|mother|father|wife|partner)" 
        r"|the\s+[^\s,;.]+(?:\s+of\s+" + name_or_gap + r")?"
        r"|(?:younger|elder|older)\s+(?:son|daughter)"
        r"|[^\s,;]+'s\s+(?:younger|elder|older)?\s*(?:son|daughter|" + occupations + r")"
        r"|(?:s\.|son|\(grand\)son|grandson|brother|sister|mother|father|daughter|wife|" + occupations + r")\s+of\s+" + gap_token + name_chars +
        r"))*"
    )


    conj_eng = (
        r"(?:"
        r"(?:,\s*(?!(?:and|ù|by)\s|<[^>]+>)" + name_chars + r")*"   # stop before gap tokens
        r"(?:,?\s*(?:and|ù)\s+(?!witnessed\s+by|by\s)" + name_chars + r")?"
        r")"
    )

    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife)\s+)?"
    
    eng_unit = (
        rf"witnessed\s+by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}{name_chars})"  
        rf"{gap_token}{relation_eng}{conj_eng}"
    )
    
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_or_gap + r")?"

    eng_pattern = (
        rf"{eng_unit}"
        rf"(?:[\s,;]+(?:and\s+)?(?:witnessed\s+)?by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}(?:(?![,;]){name_chars})?)"  
        rf"{gap_token}{relation_eng}{conj_eng}{gap_token})*"
        rf"{trailing_collective}{gap_token}[,;]?"
    )


    boundary = r'(?:[.,;:"\u201C\u201D]|<[^>]*>)'
    eng_full = rf'(^\s*|\n\s*|[.,;:"\u201C\u201D]\s*|<[^>]*>\s*)(?=witnessed\s+by)({eng_pattern})'
    try:
        highlighted_text = re.sub(
            rf"({akk_pattern})",
            lambda m: f'<span style="{blue_style}">{m.group(1)}</span>',
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )
        highlighted_text = re.sub(
            eng_full,
            lambda m: m.group(1) + f'<span style="{blue_style}">{m.group(2)}</span>',
            highlighted_text,
            flags=re.IGNORECASE | re.MULTILINE
        )
        parts = re.split(r'(<span[^>]*>.*?</span>)', highlighted_text, flags=re.DOTALL)
        result_parts = []
        for part in parts:
            if part.startswith('<span'):
                result_parts.append(part)
            else:
                escaped = html.escape(part, quote=False)  
                result_parts.append(re.sub(
                    r'(witnessed\s+by)',
                    rf'<span style="{red_style}">\1</span>',
                    escaped,
                    flags=re.IGNORECASE
                ))
        highlighted_text = ''.join(result_parts)
        highlighted_text = merge_close_spans(highlighted_text, blue_style, green_style,gap_size=5)
        highlighted_text = re.sub(r'<(;?gap[^>]*)>', r'&lt;\1&gt;', highlighted_text)
        display(HTML(f"<div style='line-height: 1.8; font-family: sans-serif; white-space: pre-wrap;'>{highlighted_text}</div>"))
    except re.error as e:
        print(f"Regex Error: {e}")

In [ ]:
def visualize_dataframe_rows(df, translit_col, transl_col,highlight_fun,substring):
    """
    Filters for rows containing 'seal of', then displays 
    the highlighted Akkadian and English side-by-side.
    """
    mask = df[transl_col].str.contains(substring, case=False, na=False)
    filtered_df = df[mask]
    
    print(f"Found {len(filtered_df)} rows containing '{substring}'.")
    
    for index, row in filtered_df.iterrows():
        print(f"\n--- Row Index: {index} ---")
        
        # Display Transliteration
        print("Akkadian:")
        highlight_fun(str(row[translit_col]))
        
        # Display Translation
        print("English:")
        highlight_fun(str(row[transl_col]))
        print("-" * 50)


In [38]:
visualize_dataframe_rows(df, Columns.TRANSLITERATION, Columns.TRANSLATION,highlight_seal_of,"seal of")

Found 172 rows containing 'seal of'.

--- Row Index: 0 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 3 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 6 ---
Akkadian:


...

--------------------------------------------------

--- Row Index: 1533 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 1547 ---
Akkadian:


English:


--------------------------------------------------


In [473]:
visualize_dataframe_rows(df, Columns.TRANSLITERATION, Columns.TRANSLATION,highlight_sealed_by,"sealed by")

Found 87 rows containing 'sealed by'.

--- Row Index: 32 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 38 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 39 ---
Akkadian:


...

--------------------------------------------------

--- Row Index: 1525 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 1544 ---
Akkadian:


English:


--------------------------------------------------


In [661]:
visualize_dataframe_rows(df, Columns.TRANSLITERATION, Columns.TRANSLATION,highlight_witnessed_by,"witnessed by")

Found 236 rows containing 'witnessed by'.

--- Row Index: 7 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 11 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 21 ---
Akkadian:


...

--------------------------------------------------

--- Row Index: 1558 ---
Akkadian:


English:


--------------------------------------------------

--- Row Index: 1560 ---
Akkadian:


English:


--------------------------------------------------


In [4]:
def build_my_patterns_witness():
    import re
    name_chars = r"[^\s,]+"
    conj_akk   = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    gap_token  = r"(?:\s*<[^>]*>(?:\s+(?!IGI\b|DUMU\b|KIŠIB\b|KÙ\.BABBAR\b|witnessed\b|by\b)[^\s<]+)?)*"
    relation_akk = (
        r"(?:\s+(?:DUMU|a-ḫu-ú)\s+[^\s,;<]+(?:\s*<[^>]*>)*"
        r"(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|a-ḫa-ma\b|a-na\b|i-na\b|§)[^\s,;]*-[^\s,;]+"
        r"(?:\s*<[^>]*>)*)*)*"
    )
    akk_name = r"[^\s]*[-\.][^\s]*[-\.][^\s]+|[^\s]*-[^\s]+(?:\s+(?!DUMU\b|IGI\b|a-ḫu-ú\b|§)[^\s]*-[^\s]+)*|[^\s]*\.[^\s]+"
    akk_igi_unit   = rf"IGI\s+(?!GÍR\b|a-we-li\b)(?:(?:{akk_name}){gap_token}(?:{relation_akk})|(?:DUMU|a-ḫu-ú)\s+{name_chars}{gap_token}{relation_akk}){conj_akk}\s*(?!\d)"
    akk_igi_multi  = rf"(?:{akk_igi_unit}\s*){{2,}}"
    akk_igi_single = rf"{akk_igi_unit}$"
    akk_igi        = rf"(?:{akk_igi_multi}|{akk_igi_single})"
    akk_kisib      = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    akk_pattern    = rf"(?:{akk_kisib}|{akk_igi})"
    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)

    occupations        = r"(?:packer|scribe|merchant|smith|weaver|potter|herald|priestess)"
    name_or_gap        = rf"(?:{name_chars}|{gap_token})"
    possessive_witness = r"(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife|partner)"
    relation_eng = (
        r"(?:[,\s]+(?:"
        r"(?:his|her)\s+(?:brother|sister|son|daughter|mother|father|wife|partner)"
        r"|the\s+[^\s,;.]+(?:\s+of\s+" + name_or_gap + r")?"
        r"|(?:younger|elder|older)\s+(?:son|daughter)"
        r"|[^\s,;]+'s\s+(?:younger|elder|older)?\s*(?:son|daughter|" + occupations + r")"
        r"|(?:s\.|son|\(grand\)son|grandson|brother|sister|mother|father|daughter|wife|" + occupations + r")\s+of\s+" + gap_token + name_chars +
        r"))*"
    )
    conj_eng = (
        r"(?:"
        r"(?:,\s*(?!(?:and|ù|by)\s|<[^>]+>)" + name_chars + r")*"
        r"(?:,?\s*(?:and|ù)\s+(?!witnessed\s+by|by\s)" + name_chars + r")?"
        r")"
    )
    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister|mother|wife)\s+)?"
    eng_unit = (
        rf"witnessed\s+by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}{name_chars})"
        rf"{gap_token}{relation_eng}{conj_eng}"
    )
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_or_gap + r")?"
    eng_pattern = (
        rf"{eng_unit}"
        rf"(?:[\s,;]+(?:and\s+)?(?:witnessed\s+)?by\s+"
        rf"(?:{possessive_witness}|{possessive_prefix}(?:(?![,;]){name_chars})?)"
        rf"{gap_token}{relation_eng}{conj_eng}{gap_token})*"
        rf"{trailing_collective}{gap_token}[,;]?"
    )
    eng_full = rf'(^\s*|\n\s*|[.,;:"\u201C\u201D]\s*|<[^>]*>\s*)(?=witnessed\s+by)({eng_pattern})'
    eng_re = re.compile(eng_full, re.IGNORECASE | re.MULTILINE)

    return akk_re, eng_re


def build_my_patterns_sealed_by():
    name_chars = r"[^\s,]+"
    
# --- AKKADIAN (no line anchor) ---
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")*"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"
    # --- ENGLISH ---
    relation_eng = r"(?:[,\s]+(?:(?:his|her)\s+(?:brother|sister)|the\s+scribe|(?:s\.|son|brother|sister|daughter|wife|\(grand\)son|grandson)\s+of\s+" + name_chars + r"))?"    
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!sealed\s+by)" + name_chars + r")?"
    #eng_unit     = rf"sealed\s+by\s+{name_chars}{relation_eng}{conj_eng}"
    trailing_collective = r"(?:,\s*the\s+(?:sons|daughters|brothers|sisters)\s+of\s+" + name_chars + r")?"
    #eng_pattern  = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?(?:sealed\s+)?by(?:\s+(?![,;]){name_chars})?{relation_eng}{conj_eng})*{trailing_collective}[,;]?"    

    possessive_prefix = r"(?:(?:his|her)\s+(?:son|daughter|brother|sister)\s+)?"
    eng_unit = rf"sealed\s+by\s+{possessive_prefix}{name_chars}{relation_eng}{conj_eng}"
    eng_pattern = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?(?:sealed\s+)?by\s+{possessive_prefix}(?:(?![,;]){name_chars})?{relation_eng}{conj_eng})*{trailing_collective}[,;]?"

    eng_full = rf"(^\s*|[,;:]\s*)({eng_pattern})"

    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)

    eng_re = re.compile(eng_full, re.IGNORECASE | re.MULTILINE)

    return akk_re, eng_re

def build_my_patterns_seal_of():
    name_chars   = r"[^\s]+"
    conj_akk     = r"(?:\s+(?:ù|and)\s+(?!KIŠIB)" + name_chars + r")?"
    relation_akk = r"(?:\s+(?:DUMU|a-ḫu-ú)\s+" + name_chars + r")?"
    akk_pattern  = rf"(?:KIŠIB\s+{name_chars}{relation_akk}{conj_akk}\s*)+"

    relation_eng = r"(?:[,\s]+(?:s\.|son|brother|sister|daughter|wife)\s+of\s+" + name_chars + r")?"
    conj_eng     = r"(?:\s+(?:and|ù)\s+(?!seal\s+of)" + name_chars + r")?"
    eng_unit     = rf"seal\s+of\s+{name_chars}{relation_eng}{conj_eng}"
    eng_pattern  = rf"{eng_unit}(?:[\s,;]+(?:and\s+)?seal\s+of(?:\s+(?![,;]){name_chars})?{relation_eng}{conj_eng})*[,;]?"
    eng_full     = rf"(^|[.,;:]\s*)({eng_pattern})"

    akk_re = re.compile(rf"({akk_pattern})", re.IGNORECASE | re.MULTILINE)
    eng_re = re.compile(eng_full,            re.IGNORECASE | re.MULTILINE)
    return akk_re, eng_re

def build_my_patterns_by_hand():
    match_all_akk = re.compile(r'(.+)', re.DOTALL)
    match_all_eng = re.compile(r'()(.+)', re.DOTALL)  # group 1 = empty prefix, group 2 = everything

    return match_all_akk, match_all_eng

In [6]:

akk_re, eng_re = build_my_patterns_by_hand()  # or your own compiled re.compile(...)
clean, flagged = process_dataframe(
    df,
    translit_col=Columns.TRANSLITERATION,
    transl_col=Columns.TRANSLATION,
    substring='Seal of',          # matches if found in Akkadian OR English
    akk_pattern=akk_re, akk_group=1,
    eng_pattern=eng_re, eng_group=2,
    ratio_min=0.58,   # override here
    ratio_max=1.4,   # override here
    pos_threshold = 0.15
)

Rows matching 'Seal of': 172
Processed 172 rows — 150 clean  |  22 flagged


In [5]:
# Step 2 — review flagged items interactively
reviewer = BatchReviewer(flagged)
reviewer.show()

In [48]:
# Step 3 — after review, collect final aligned segments
all_results = clean + flagged
dataset = build_final_dataset(all_results)  # list of {'row_index', 'segment_idx', 'akk', 'eng'}

In [49]:
df_aligned = pd.DataFrame(dataset)

In [50]:
df_aligned.to_csv("seal_of.csv")

In [ ]:
seal_of   = pd.read_csv("seal_of.csv")
sealed_by = pd.read_csv("sealed_by.csv")
witness   = pd.read_csv("witness.csv")

In [33]:

def remove_short_rows(df, translit_col, transl_col, min_chars=5):
    total_chars = df[translit_col].fillna('').str.len() + df[transl_col].fillna('').str.len()
    return df[total_chars > min_chars].reset_index(drop=True)

def remove_nan_ratio_rows(df, translit_col, transl_col, max_ratio=20):
    def has_nan_word(text):
        return bool(re.search(r'\bnan\b', str(text), re.IGNORECASE))

    def word_ratio(a, b):
        wa = len(str(a).split())
        wb = len(str(b).split())
        if wb == 0 or wa == 0:
            return float('inf')
        return max(wa, wb) / min(wa, wb)  # always >= 1

    mask = df.apply(
        lambda row: (
            has_nan_word(row[translit_col]) or has_nan_word(row[transl_col])
        ) and word_ratio(row[translit_col], row[transl_col]) > max_ratio,
        axis=1
    )
    return df[~mask].reset_index(drop=True)



In [ ]:
df_extracted = pd.concat([seal_of, sealed_by, witness], ignore_index=True)
print(f"Before cleaning: {len(df_extracted)}")
df_extracted = remove_short_rows(df_extracted, "akk", "eng")
print(f"After short cleaning: {len(df_extracted)}")
df_extracted = remove_nan_ratio_rows(df_extracted, "akk", "eng")
print(f"After nan/ratio cleaning: {len(df_extracted)}")

In [6]:

akk_re, eng_re = build_my_patterns_by_hand()  # or your own compiled re.compile(...)
clean, flagged = process_dataframe(
    df_extracted,
    translit_col="akk",
    transl_col="eng",
    substring='',          # matches if found in Akkadian OR English
    akk_pattern=akk_re, akk_group=1,
    eng_pattern=eng_re, eng_group=2,
    ratio_min=0.58,   # override here
    ratio_max=1.4,   # override here
    pos_threshold = 0.15
)

Processed 878 rows — 791 clean  |  87 flagged


In [7]:
# Step 2 — review flagged items interactively
reviewer = BatchReviewer(flagged)
reviewer.show()

In [10]:
# Step 3 — after review, collect final aligned segments
all_results = clean + flagged
dataset = build_final_dataset(all_results)  # list of {'row_index', 'segment_idx', 'akk', 'eng'}
dataset = pd.DataFrame(dataset)

In [11]:
dataset.to_csv("joined_cleaned.csv")

In [12]:
print(f"Before cleaning: {len(dataset)}")
dataset = remove_short_rows(dataset, "akk", "eng")
print(f"After short cleaning: {len(dataset)}")
dataset = remove_nan_ratio_rows(dataset, "akk", "eng")
print(f"After nan/ratio cleaning: {len(dataset)}")


Before cleaning: 873
After short cleaning: 873
After nan/ratio cleaning: 873


In [14]:
akk_re_seal,   eng_re_seal   = build_my_patterns_seal_of()
akk_re_sealed, eng_re_sealed = build_my_patterns_sealed_by()
akk_re_wit,    eng_re_wit    = build_my_patterns_witness()

common = dict(
    df           = df,
    translit_col = Columns.TRANSLITERATION,
    transl_col   = Columns.TRANSLATION,
    ratio_min    = 0.58,
    ratio_max    = 1.4,
    pos_threshold= 0.15,
    id_col       = 'id',
)

clean_seal,   flagged_seal   = process_dataframe(**common, substring='Seal of',     akk_pattern=akk_re_seal,   akk_group=1, eng_pattern=eng_re_seal,   eng_group=2)
clean_sealed, flagged_sealed = process_dataframe(**common, substring='Sealed by',   akk_pattern=akk_re_sealed, akk_group=1, eng_pattern=eng_re_sealed, eng_group=2)
clean_wit,    flagged_wit    = process_dataframe(**common, substring='witnessed by', akk_pattern=akk_re_wit,    akk_group=1, eng_pattern=eng_re_wit,    eng_group=2)

df_seal   = pd.DataFrame(build_final_dataset(clean_seal   + flagged_seal))
df_sealed = pd.DataFrame(build_final_dataset(clean_sealed + flagged_sealed))
df_wit    = pd.DataFrame(build_final_dataset(clean_wit    + flagged_wit))

df_new_segments   = pd.concat([df_seal, df_sealed, df_wit], ignore_index=True)
all_processed_ids = df_new_segments['source_id'].dropna().unique()

df_original_cleaned = df[~df['id'].isin(all_processed_ids)]
print()
print(f"Original size: {len(df)}")
print(f"New segments created : {len(df_new_segments)}")
print(f"Original rows removed: {len(all_processed_ids)}")
print(f"Remaining original   : {len(df_original_cleaned)}")
print(f"New dataset size: {len(df_new_segments) + len(df_original_cleaned)}")

Rows matching 'Seal of': 172
Processed 153 rows — 122 clean  |  31 flagged
Rows matching 'Sealed by': 87
Processed 37 rows — 36 clean  |  1 flagged
Rows matching 'witnessed by': 236
Processed 212 rows — 148 clean  |  64 flagged

Original size: 1561
New segments created : 910
Original rows removed: 402
Remaining original   : 1159
New dataset size: 2069


In [24]:
pre_clean_df = pd.read_csv("full_cleaned2.csv")

In [25]:

akk_re, eng_re = build_my_patterns_by_hand()  # or your own compiled re.compile(...)
clean, flagged = process_dataframe(
    pre_clean_df,
    translit_col="akk",
    transl_col="eng",
    substring='',          # matches if found in Akkadian OR English
    akk_pattern=akk_re, akk_group=1,
    eng_pattern=eng_re, eng_group=2,
    ratio_min=0.4,   # override here
    ratio_max=3,   # override here
    pos_threshold = 0.15
)

Processed 1157 rows — 1069 clean  |  88 flagged


In [26]:
# Step 2 — review flagged items interactively
reviewer = BatchReviewer(flagged)
reviewer.show()

In [37]:
# Open the file in write mode ('w')
with open("example.txt", "w") as file:
    file.write(str(flagged))

In [27]:
# Step 3 — after review, collect final aligned segments


all_results = clean + flagged
dataset_2 = pd.DataFrame(build_final_dataset(all_results,mode='matches_only'))  # list of {'row_index', 'segment_idx', 'akk', 'eng'}

In [30]:
dataset_2.to_csv("full_cleaned4.csv")

In [31]:
new_dataset = pd.read_csv("full_cleaned4.csv")

In [34]:
print(f"Before cleaning: {len(new_dataset)}")
new_dataset = remove_short_rows(new_dataset, "akk", "eng")
print(f"After short cleaning: {len(new_dataset)}")
new_dataset = remove_nan_ratio_rows(new_dataset, "eng", "eng")
print(f"After nan/ratio cleaning: {len(new_dataset)}")

Before cleaning: 1157
After short cleaning: 1157
After nan/ratio cleaning: 1157


In [25]:
dataset.to_csv("full_cleaned2.csv")

In [39]:
df1 = pd.read_csv("sealed_seal_witness_cleaned.csv")
df2 = pd.read_csv("full_cleaned4.csv")

# 2. Concatenate the dataframes
# ignore_index=True resets the index so it flows from 0 to N
combined_df = pd.concat([df1, df2], ignore_index=True)

# 3. Save to a new CSV file
# index=False prevents pandas from adding an extra column for the row numbers
combined_df.to_csv("train-v1.csv", index=False)